# Session 06 Topic 04: Comparing groups by their intervals

Use this notebook while working through Topic 04. This is the point of the whole
session.

Clients rarely ask what a mean is. They ask whether one group **differs** from
another. The instinctive move is to compare two averages and report whichever is
bigger — and that move is wrong often enough to be dangerous, because two averages
always differ by *something*.

You will build one interval per group, plot them side by side, and learn to read three
different situations correctly.

Charts are supplied. You will write the interval calculations, run the comparisons, and
answer the activity questions.

## 1. Setup and the question

A transport planner has noticed that Mondays have felt quieter since 2020 and wants to
know whether the data support moving services off Mondays.

> On the Sandringham line in 2023-24, does the mean number of boardings differ across
> the five weekdays?

Note that the question came **first**, before any calculation. Topic 05 explains why
that ordering matters far more than it looks.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="colorblind")

DATA_FOLDER = Path("data")
TRAIN_FILE = DATA_FOLDER / "train_daily_boardings.csv"

train = pd.read_csv(TRAIN_FILE, parse_dates=["business_date"])

sandringham = train[
    (train["line_name"] == "Sandringham")
    & (train["day_type"] == "Normal Weekday")
    & (train["financial_year"] == "2023-24")
]

WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

print("Days:", len(sandringham))

## 2. One interval per group

The calculation is the same as Topic 03, just repeated once per group. Write it as a
function so it can be reused later in the notebook — you will need it three times.

In [ ]:
# Write a function called interval_table(data, group_column, groups).
# Start with an empty list called rows.
# Loop over each group in groups:
#   - pick out the total_boardings values for that group
#   - work out n, the mean and the standard error
#   - get the interval with stats.t.interval
#   - append a dictionary with group, n, mean, low and high to rows
# Return pd.DataFrame(rows).
# 
# Then call it on sandringham, grouping by day_of_week over WEEKDAYS.

### Activity — Read the table first


Before looking at the chart, use only the table. Which day is clearly different from
the others? Which days do you think cannot be told apart?

**Your answer:** Double-click this cell and replace this text with your response.

## 3. The chart

Numbers in a table are hard to compare. The same information as a picture is not.

The plotting code is supplied as a function, because you will reuse it.

In [ ]:
def plot_intervals(summary, title, highlight=None):
    """Draw one horizontal interval per group, newest at the top."""
    if highlight is None:
        highlight = []

    plt.figure(figsize=(8.5, 4.2))

    # One row per group, counting down so the first group appears at the top.
    positions = range(len(summary) - 1, -1, -1)

    for position, row in zip(positions, summary.itertuples()):
        colour = "#D55E00" if row.group in highlight else "#0072B2"

        # The interval itself.
        plt.plot([row.low, row.high], [position, position], color=colour, linewidth=2.5)

        # Small caps at each end.
        plt.plot([row.low, row.low], [position - 0.12, position + 0.12], color=colour, linewidth=2.5)
        plt.plot([row.high, row.high], [position - 0.12, position + 0.12], color=colour, linewidth=2.5)

        # A dot at the mean.
        plt.plot(row.mean, position, "o", color=colour, markersize=8)

    plt.yticks(list(positions), summary["group"])
    plt.title(title)
    plt.xlabel("Mean daily boardings (95% confidence interval)")
    plt.show()


plot_intervals(
    weekday_summary,
    "Sandringham 2023-24: mean daily boardings by weekday",
    highlight=["Monday"],
)

The chart answers the planner's question immediately, and it answers it in two parts.

**Monday is genuinely lower.** Its interval ends at 36,538; the next lowest interval
begins at 40,050. There is a gap of thousands of journeys between them. No plausible
population mean for Monday comes anywhere close to the others.

**Tuesday to Friday cannot be separated.** Their intervals overlap substantially.
Thursday has the highest average, but Thursday's interval comfortably contains
Tuesday's mean and vice versa. The data does not support ranking them.

The second finding is as valuable as the first. Comparing point estimates alone, the
planner might have concluded that Thursday is 2,500 journeys busier than Tuesday and
rostered accordingly — a difference the data cannot actually establish.

### Activity — Answer the planner.


Write two sentences for the planner: one on what the data show about Mondays, and
one on what it cannot support about the rest of the week.

**Your answer:** Double-click this cell and replace this text with your response.

## 4. The three cases

Two intervals can do three things, and each licenses a different conclusion. The chart
below picks one real pair from this dataset to show each.

In [ ]:
cases = [
    ("Clear gap", "Monday", "Thursday"),
    ("Heavy overlap", "Tuesday", "Wednesday"),
    ("Slight overlap", "Tuesday", "Thursday"),
]

fig, axes = plt.subplots(3, 1, figsize=(8.5, 6), sharex=True)

for axis, (label, first_day, second_day) in zip(axes, cases):
    for position, day in enumerate([first_day, second_day]):
        row = weekday_summary[weekday_summary["group"] == day].iloc[0]

        axis.plot([row["low"], row["high"]], [position, position], linewidth=3, color="#0072B2")
        axis.plot(row["mean"], position, "o", color="#0072B2", markersize=8)
        axis.text(row["high"] + 200, position, " " + day, va="center", fontsize=9)

    axis.set_yticks([])
    axis.set_ylim(-0.6, 1.6)
    axis.set_title(label, loc="left", fontsize=11)

axes[-1].set_xlabel("Mean daily boardings (95% confidence interval)")
plt.tight_layout()
plt.show()

**Clear gap — the groups differ:** Monday against Thursday. The intervals do not come
close to touching, so every plausible value for one lies below every plausible value
for the other. Report the difference with confidence.

**Heavy overlap — the data cannot separate them.** Tuesday against Wednesday. Each
interval contains the other's mean. The honest report is "no detectable difference",
which is *not* the same as "no difference".

**Slight overlap — undecided.** Tuesday against Thursday. The intervals graze each
other. Later we will show the tool that will help in cases like this.

Practically, this gives you a three-way rule:

| What you see | What you may conclude |
|---|---|
| clear gap | strong evidence that the groups differ |
| heavy overlap | the data cannot separate them |
| slight overlap | undecided — investigate further |

## 6. Resolving the undecided case

For the third row we need a direct comparison of the difference. The tool is a
**t-test**, and Python runs it in one line.

The number it returns is a **p-value**, which answers one narrow question:

> If the two population means were equal, how compatible would a gap at least this
> large be with random sampling variation?

A small p-value says a gap this large would be unusual if the population means were
equal, so it provides evidence of a difference. A large p-value means the data does not
establish a difference; it does not prove that the means are equal.

In [ ]:
# Pull out the total_boardings for Tuesday and for Thursday.
# Run stats.ttest_ind on the two, with equal_var=False.
# Print both means and the p-value, rounded to 4 decimal places.

Now run it on every pair, so the three cases can be seen together.

In [ ]:
from itertools import combinations

comparisons = []

for first_day, second_day in combinations(WEEKDAYS, 2):
    first = sandringham.loc[sandringham["day_of_week"] == first_day, "total_boardings"]
    second = sandringham.loc[sandringham["day_of_week"] == second_day, "total_boardings"]

    first_row = weekday_summary[weekday_summary["group"] == first_day].iloc[0]
    second_row = weekday_summary[weekday_summary["group"] == second_day].iloc[0]

    # Do the two intervals share any ground?
    overlap = not (first_row["high"] < second_row["low"] or second_row["high"] < first_row["low"])

    p_value = stats.ttest_ind(first, second, equal_var=False).pvalue

    comparisons.append({
        "pair": first_day + " v " + second_day,
        "gap": abs(round(first.mean() - second.mean())),
        "intervals_overlap": overlap,
        "p_value": round(p_value, 4),
    })

pd.DataFrame(comparisons)

Read that table against the three cases:

- Every **Monday** comparison has no overlap and a p-value below 0.0001. The chart had
  already settled these.
- **Tuesday v Wednesday** overlaps heavily, p = 0.42. No detectable difference.
- **Tuesday v Thursday** and **Thursday v Friday** overlap slightly, with p = 0.036 in
  both cases. These provide evidence of differences the chart could not resolve.

The convention is to treat p below **0.05** as evidence of a difference. Read that
as a guide rather than a verdict — 0.049 and 0.051 are not meaningfully different, and
neither is a proof of anything.

> **Choose the tool for the audience.** For a visual explanation or a lay audience, the
> chart is the best tool. It shows the size of each mean, the precision of each estimate,
> and every group at once.

For research using this method, p-values are always reported for formal comparisons
and should appear alongside the chart. This is only the beginning of a larger topic
called **hypothesis testing**. Formal hypotheses, assumptions, error types, and study
design are beyond the scope of this unit.

### Activity — Resolve a pair


Thursday and Friday also overlap slightly. Predict the outcome, then find that pair in
the table above. Was the gap resolvable?

**Your answer:** Double-click this cell and replace this text with your response.

## 7. A second question: did patronage recover?

The same method answers a question over time. Patronage collapsed in 2020, and a
planner wants to know when recovery actually began.

This is where the function you wrote earns its keep — the group column changes, but
nothing else does.

In [ ]:
# Filter train to Williamstown normal weekdays - all six years, no year filter.
# Make a list of the six financial years.
# Call interval_table on it, grouping by financial_year.

In [ ]:
plot_intervals(
    recovery,
    "Williamstown: mean weekday boardings by financial year",
    highlight=["2020-21", "2021-22"],
)

In [ ]:
year_2020 = williamstown.loc[williamstown["financial_year"] == "2020-21", "total_boardings"]
year_2021 = williamstown.loc[williamstown["financial_year"] == "2021-22", "total_boardings"]

print("2020-21 mean:", round(year_2020.mean()))
print("2021-22 mean:", round(year_2021.mean()))
print("Difference:  ", round(year_2021.mean() - year_2020.mean()))
print("p-value:     ", round(stats.ttest_ind(year_2020, year_2021, equal_var=False).pvalue, 4))

### Activity — Write the finding


Write two sentences for a report: one about the 2020-21 to 2021-22 comparison, and one
about 2022-23. Avoid claiming more than the intervals support.

**Your answer:** Double-click this cell and replace this text with your response.

## 8. The same question, a different line

Findings hold for the data they came from. Run the weekday analysis on Alamein.

In [ ]:
# Filter train to Alamein, Normal Weekday, 2023-24.
# Build its weekday interval table with your interval_table function.
# Plot it with plot_intervals, highlighting Monday and Friday.
# Show the table too.

In [ ]:
alamein_monday = alamein.loc[alamein["day_of_week"] == "Monday", "total_boardings"]
alamein_friday = alamein.loc[alamein["day_of_week"] == "Friday", "total_boardings"]

print("Alamein Monday mean:", round(alamein_monday.mean()))
print("Alamein Friday mean:", round(alamein_friday.mean()))
print("p-value:            ", round(stats.ttest_ind(alamein_monday, alamein_friday, equal_var=False).pvalue, 4))

Alamein has a **two-day** pattern. Friday is as quiet as Monday, and the two cannot be
separated from each other (p = 0.65) — though both sit clearly below Tuesday, Wednesday
and Thursday.

A conclusion drawn from Sandringham alone — "Mondays are quiet" — would have been right
about Sandringham and incomplete about the network.

## What you have done

- Written a reusable function that produces one interval per group
- Plotted intervals side by side and read a real question off the chart
- Learnt the three cases: **clear gap**, **heavy overlap**, **slight overlap**
- Recognised that a clear gap is strong visual evidence, while a slight overlap leaves the
  comparison undecided
- Used a t-test and p-value to resolve two pairs the chart left undecided
- Identified charts as the best tool for a lay audience and p-values as part of formal
  research reporting for this method
- Found a difference that was not there (Williamstown 2020-21 to 2021-22) and a
  pattern that did not generalise (Alamein)

**Next:** Topic 05's notebook applies all of this to student results, where the
judgements get harder.